# TN_test — dựng lại NGUYÊN BẢN kiến trúc TCN do Optuna chọn

## Vì sao có notebook này

Notebook `TN_test_ds_tcn_192.ipynb` **chưa dựng đúng** kiến trúc đó. Nó
dùng `--kernel_size 5` và giữ BatchNorm của code hiện tại, nên khác bản gốc ở
hai chỗ. Notebook này dựng lại đúng.

## Cấu hình

| thành phần | giá trị |
|---|---|
| channels | **192** và **64** |
| số khối | 4, mỗi khối hai phép tích chập depthwise separable |
| kernel / độ giãn | 3 / 1, 2, 4, 8 |
| tầm nhìn | **61 mẫu** |
| chuẩn hoá | **không có** |
| dropout | `nn.Dropout(0.2)` |
| vào / ra | 200 → 25 |
| RevIN | tắt |
| tham số | **307.801** (c192) và **37.081** (c64) |

## Hai chỗ code hiện tại chưa làm được, đã bổ sung

Kiến trúc đó khác `ds_tcn` của TN1 đúng hai điểm, và cả hai đều phải thêm tuỳ
chọn mới:

**`--norm none`.** Code cũ chỉ nhận `batch` hoặc `weight`. Model của thí nghiệm
cũ **không có lớp chuẩn hoá nào** — đó chính là 3.072 tham số chênh lệch đã tìm
ra trước đây (8 lớp BatchNorm × 384).

**`--dropout_kind element`.** TN1 dùng `nn.Dropout1d`, xoá cả một kênh, theo
"spatial dropout" ở Bai mục 3.4. Thí nghiệm cũ dùng `nn.Dropout` thường, xoá
từng phần tử. Hai loại chỉ khác nhau khi `dropout > 0`, mà mọi cấu hình TN1 đều
để 0, nên chưa từng lộ ra.

## Đã đối chiếu với bản gốc, khớp từng bit

Dựng lại nguyên văn lớp `CausalDSConv`, `TCNBlock`, `ForecastTCN` của vòng dò,
chép trọng số sang bản này, so đầu ra:

```
tham số   cũ 307801   mới 307801   khớp True
cùng số tensor: True | cùng hình dạng: True
đầu ra giống hệt: True | chênh lớn nhất: 0.000e+00
```

Nên đây là **đúng kiến trúc đó**, không phải bản gần đúng.

## Tầm nhìn 61 — đọc kỹ chỗ này

Cửa sổ đầu vào 200 mẫu, nhưng model chỉ thấy **61 mẫu gần nhất**. Nó bỏ qua
**70%** đầu cửa sổ.

    TN1 ds_tcn      kernel 3, 6 khối  -> tầm nhìn 253, phủ trọn 200
    Optuna chọn     kernel 3, 4 khối  -> tầm nhìn  61, thấy 61/200

Ở 50 Hz thì 61 mẫu là **1,2 giây**, trong khi một nhịp thở khoảng 4 giây. Nghĩa
là model không nhìn đủ một chu kỳ thở.

Thí nghiệm cũ có thử cấu hình tầm nhìn dài hơn (RF121, kernel 5) và thấy nó tốt
hơn trên GHIJ: 0,8086 so với 0,7950. Nhưng cấu hình Optuna chọn lại là bản tầm
nhìn 61. Đây là chỗ đáng bàn khi báo cáo.

## Hai cấu hình chạy ở đây

| | channels | tham số | so với |
|---|---:|---:|---|
| A | 192 | 307.801 | cấu hình Optuna chọn |
| B | 64 | 37.081 | cùng kiến trúc, nhỏ hơn 8,3 lần |

B là đối chứng: nếu A hơn B rõ thì số kênh có tác dụng ở kiến trúc này; nếu
ngang nhau thì kết luận "sức chứa đã bão hoà" được củng cố thêm một điểm.

Cả hai chạy **một seed**, ghi vào `runs/tn_test/`. Giao thức giữ nguyên TN1:
20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9, bốn fold cũ. **Không** dùng
learning rate và weight decay mà Optuna dò được — giữ nguyên giao thức thì mới
so được với các cấu hình khác.

Ước lượng: A khoảng **2 giờ**, B khoảng **45 phút**.

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn. Hai tuỳ chọn `--norm none` và `--dropout_kind element` chỉ có
ở bản mới, nên ô này phải chạy được thì mới đi tiếp.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm hai bản cài đặt

Bảy phép kiểm mỗi cấu hình. Ba phép mới nhắm đúng những chỗ vừa bổ sung:

    mục 5   KHÔNG có lớp chuẩn hoá nào — đúng bản gốc
    mục 6   dùng nn.Dropout chứ không phải nn.Dropout1d
    mục 7   in tầm nhìn 61 và cảnh báo mất 70% đầu cửa sổ

Số tham số phải ra đúng **307.801** và **37.081**.

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 192 --kernel_size 3 \
    --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

In [ ]:
!python scripts/check_model.py --model ds_tcn --channels 64 --kernel_size 3 \
    --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

## 3. A — 192 kênh, 307.801 tham số

Tên cấu hình: `ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0`.

Hậu tố `_none` ghi cách chuẩn hoá, `_dpel` ghi loại dropout. Cả hai chỉ xuất
hiện khi khác mặc định, nên tên của mọi lần chạy cũ không đổi — đã đối chiếu
từng ký tự cho cả 13 cấu hình đang có.

Khoảng **2 giờ**.

In [ ]:
!python scripts/run_cv.py --experiment tn_test --model ds_tcn --channels 192 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

## 4. B — 64 kênh, 37.081 tham số

Cùng kiến trúc, chỉ đổi số kênh. Khoảng **45 phút**.

In [ ]:
!python scripts/run_cv.py --experiment tn_test --model ds_tcn --channels 64 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

## 5. Cất kết quả

In [ ]:
!python scripts/save_results.py tn_test --out tn_test_tcn_optuna

## 6. Bảng so

`runs/tn_test/` giờ có ba cấu hình: hai cái vừa chạy, và
`ds_tcn_c192_k5_n4_do0.2` từ lần trước (bản chưa dựng đúng, có BatchNorm, tầm
nhìn 121).

Mốc từ TN1 để đặt cạnh:

| | tham số | tầm nhìn | cv_mean |
|---|---:|---:|---:|
| LSTM-352 | 1.502.713 | — | 0,7570 |
| LSTM-67 | 56.908 | — | 0,7532 |
| DS-TCN-64 (TN1) | 56.281 | 253 | 0,7421 |
| DS-TCN-192 k5 | 313.945 | 121 | 0,7566 *(1 seed)* |

**Nhớ ngưỡng nhiễu.** Tám cấu hình TN1 có `seed_std` trung bình 0,0034, lớn
nhất 0,0108. Một seed thì chênh lệch dưới khoảng 0,01 chưa đọc được gì.

**Và fold `val_KL` bị nhiễm cho riêng nhóm này** — Optuna vốn chọn cấu hình
bằng cách tối ưu trên chính KL. Đọc ba fold `val_AB`, `val_CE`, `val_DF` tách
riêng.

In [ ]:
!python scripts/compare_cv.py --experiment tn_test

## 7. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()